In [0]:
# ── CONFIG ───────────────────────────────────────────────────────────────────
 
CATALOG        = "clutchlytics"
FCT_GAMES      = f"{CATALOG}.silver.fctGames"
DIM_TEAMS      = f"{CATALOG}.silver.dimTeams"
SILVER_TABLE   = f"{CATALOG}.silver.nhl_series"
 
SPORT  = "hockey"
LEAGUE = "nhl"
 
print(f"Source    : {FCT_GAMES}")
print(f"Target    : {SILVER_TABLE}")

In [0]:
# ── READ SOURCE ───────────────────────────────────────────────────────────────
 
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime, timezone
 
fct_df = (
    spark.table(FCT_GAMES)
    .filter(F.col("league") == LEAGUE)
)
 
print(f"fctGames rows ({LEAGUE}): {fct_df.count()}")
 
# ── dimTeams for abbreviation lookup ──
dim_teams = (
    spark.table(DIM_TEAMS)
    .filter(F.col("league") == LEAGUE)
    .select(
        F.col("clutch_team_id").alias("lookup_team_id"),
        F.col("abbreviation").alias("lookup_abbr"),
    )
)

# ── dimGames for season_type ──
dim_games = (
    spark.table(f"{CATALOG}.silver.dimGames")
    .filter(F.col("league") == LEAGUE)
    .select(
        F.col("source_event_id").alias("dim_event_id"),
        F.col("season_type"),
    )
)

In [0]:
# ── BUILD BASE — one row per game with team_a/team_b consistent ordering ──────────────────────────────
 
base_df = (
    fct_df
    .select(
        "source_event_id",
        "clutch_game_id",
        "clutch_home_team_id",
        "clutch_away_team_id",
        "home_team_abbr",
        "away_team_abbr",
        "home_score",
        "away_score",
        "home_winner",
        "away_winner",
        "went_to_ot",
        "home_series_wins",
        "away_series_wins",
        "series_clinched",
        "game_number_in_series",
        "series_key",
        "round",
        # season_type removed - will join from dimGames
    )
    # ── Filter to completed games with valid series context ──
    .filter(
        F.col("series_key").isNotNull() &
        F.col("game_number_in_series").isNotNull()
    )
)

# ── Join dimGames to get season_type ──
base_df = (
    base_df
    .join(
        dim_games,
        base_df.source_event_id == dim_games.dim_event_id,
        how="left"
    )
    .drop("dim_event_id")
)
 
# ── Assign team_a / team_b consistently ──
# team_a = LEAST(clutch_home_team_id, clutch_away_team_id) — always lower ID
# Mirrors the series_key derivation in dimGames for consistency
base_df = (
    base_df
    .withColumn(
        "team_a_clutch_id",
        F.least(F.col("clutch_home_team_id"), F.col("clutch_away_team_id"))
    )
    .withColumn(
        "team_b_clutch_id",
        F.greatest(F.col("clutch_home_team_id"), F.col("clutch_away_team_id"))
    )
)
 
# ── Resolve team_a and team_b abbreviations from dimTeams ──
base_df = (
    base_df
    .join(
        dim_teams.select(
            F.col("lookup_team_id").alias("ta_id"),
            F.col("lookup_abbr").alias("team_a_abbr"),
        ),
        base_df.team_a_clutch_id == F.col("ta_id"),
        how="left"
    )
    .drop("ta_id")
    .join(
        dim_teams.select(
            F.col("lookup_team_id").alias("tb_id"),
            F.col("lookup_abbr").alias("team_b_abbr"),
        ),
        base_df.team_b_clutch_id == F.col("tb_id"),
        how="left"
    )
    .drop("tb_id")
)
 
print(f"Base rows after team assignment: {base_df.count()}")

In [0]:
# ── DERIVE PER-GAME STATS ─────────────────────────────────────────────────────
 
# ── Game winner abbreviation ──
base_df = base_df.withColumn(
    "game_winner_abbr",
    F.when(F.col("home_winner") == True, F.col("home_team_abbr"))
     .when(F.col("away_winner") == True, F.col("away_team_abbr"))
     .otherwise(None)
)
 
# ── Goal differential (absolute margin) ──
base_df = base_df.withColumn(
    "goal_differential",
    F.abs(F.col("home_score") - F.col("away_score"))
)
 
# ── team_a wins and team_b wins per game
# home_series_wins and away_series_wins in fctGames are post-game standings
# Map them to team_a / team_b based on whether home team = team_a
base_df = (
    base_df
    .withColumn(
        "team_a_wins",
        F.when(
            F.col("clutch_home_team_id") == F.col("team_a_clutch_id"),
            F.col("home_series_wins")
        ).otherwise(F.col("away_series_wins"))
    )
    .withColumn(
        "team_b_wins",
        F.when(
            F.col("clutch_home_team_id") == F.col("team_b_clutch_id"),
            F.col("home_series_wins")
        ).otherwise(F.col("away_series_wins"))
    )
)
 
# ── Series leader after this game ──
base_df = base_df.withColumn(
    "series_leader_abbr",
    F.when(F.col("team_a_wins") > F.col("team_b_wins"), F.col("team_a_abbr"))
     .when(F.col("team_b_wins") > F.col("team_a_wins"), F.col("team_b_abbr"))
     .otherwise(None)   # NULL = tied
)
 
base_df = base_df.withColumn(
    "series_tied",
    F.col("team_a_wins") == F.col("team_b_wins")
)
 
# ── Cumulative goal differential ──
# Positive value favors team_a (lower clutch_team_id)
# Each game contributes +goal_differential if team_a won, -goal_differential if team_b won
base_df = base_df.withColumn(
    "game_goal_diff_signed",
    F.when(
        F.col("game_winner_abbr") == F.col("team_a_abbr"),
        F.col("goal_differential")
    ).otherwise(
        F.col("goal_differential") * -1
    )
)
 
series_window = Window.partitionBy("series_key").orderBy("game_number_in_series")
 
base_df = base_df.withColumn(
    "cumulative_goal_diff",
    F.sum("game_goal_diff_signed").over(series_window)
)
 
print("Per-game derivations complete.")
print("\nSample — first series:")
base_df.filter(
    F.col("series_key") == base_df.select("series_key").first()[0]
).select(
    "series_key", "game_number_in_series", "home_team_abbr",
    "home_score", "away_team_abbr", "away_score",
    "team_a_wins", "team_b_wins", "series_leader_abbr",
    "series_tied", "cumulative_goal_diff", "series_clinched"
).orderBy("game_number_in_series").show(truncate=False)

In [0]:
# ── FINAL COLUMN SELECTION ────────────────────────────────────────────────────
 
ingested_at = datetime.now(timezone.utc).isoformat()
 
series_df = (
    base_df
    .withColumn("sport",       F.lit(SPORT))
    .withColumn("league",      F.lit(LEAGUE))
    .withColumn("ingested_at", F.lit(ingested_at))
    .withColumn("season",      F.lit(2026))
    .select(
        # ── Series identity ──
        "series_key",
        "round",
        "season",
        "sport",
        "league",
        "season_type",
 
        # ── Teams ──
        "team_a_clutch_id",
        "team_b_clutch_id",
        "team_a_abbr",
        "team_b_abbr",
 
        # ── Per game ──
        F.col("game_number_in_series").alias("game_number"),
        "clutch_game_id",
        "source_event_id",
 
        # ── Game result ──
        "home_team_abbr",
        "away_team_abbr",
        "home_score",
        "away_score",
        "went_to_ot",
        "goal_differential",
        "game_winner_abbr",
 
        # ── Series standing after this game ──
        "team_a_wins",
        "team_b_wins",
        "series_leader_abbr",
        "series_tied",
        "series_clinched",
 
        # ── Momentum ──
        "cumulative_goal_diff",
 
        # ── Metadata ──
        "ingested_at",
        F.lit("silver.fctGames").alias("source_table"),
    )
    .orderBy("series_key", "game_number")
)
 
print(f"Total rows to write: {series_df.count()}")

In [0]:
# ── WRITE TO SILVER ───────────────────────────────────────────────────────────
# MERGE on series_key + game_number — safe for re-runs and new round uploads.
 
table_exists = spark.catalog.tableExists(SILVER_TABLE)
 
if not table_exists:
    (
        series_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(SILVER_TABLE)
    )
    print(f"Table created: {SILVER_TABLE}")
 
else:
    series_df.createOrReplaceTempView("new_series")
 
    spark.sql(f"""
        MERGE INTO {SILVER_TABLE} AS target
        USING new_series AS source
        ON  target.series_key  = source.series_key
        AND target.game_number = source.game_number
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"Merged into existing table: {SILVER_TABLE}")

In [0]:
# ── VALIDATE — show every series game by game ─────────────────────────────────
 
print("── All series — game by game ──")
spark.sql(f"""
    SELECT
        series_key,
        game_number,
        home_team_abbr,
        home_score,
        away_team_abbr,
        away_score,
        went_to_ot,
        goal_differential,
        team_a_wins,
        team_b_wins,
        series_leader_abbr,
        series_tied,
        series_clinched,
        cumulative_goal_diff
    FROM {SILVER_TABLE}
    ORDER BY series_key, game_number
""").show(100, truncate=False)

In [0]:
# ── SANITY CHECKS ─────────────────────────────────────────────────────────────
 
checks = spark.sql(f"""
    SELECT
        COUNT(*)                                                    AS total_rows,
        COUNT(DISTINCT series_key)                                  AS unique_series,
        COUNT(CASE WHEN series_clinched = true  THEN 1 END)        AS clinching_games,
        COUNT(CASE WHEN series_tied = true      THEN 1 END)        AS tied_after_game,
        COUNT(CASE WHEN went_to_ot = true       THEN 1 END)        AS ot_games,
        COUNT(CASE WHEN team_a_clutch_id IS NULL THEN 1 END)       AS null_team_a,
        COUNT(CASE WHEN team_b_clutch_id IS NULL THEN 1 END)       AS null_team_b,
        COUNT(CASE WHEN clutch_game_id IS NULL  THEN 1 END)        AS null_clutch_game_ids,
        COUNT(CASE WHEN series_leader_abbr IS NULL
                    AND series_tied = false THEN 1 END)            AS bad_leader_logic,
        MAX(game_number)                                            AS max_games_in_series,
        MIN(game_number)                                            AS min_games_in_series,
        ROUND(AVG(goal_differential), 2)                           AS avg_goal_diff
    FROM {SILVER_TABLE}
""")
 
print("Sanity checks:")
checks.show(truncate=False)